# Dragon SpaceX Automated Docking - PID Controller
Este notebook integra un controlador PID discreto de 6 grados de libertad para acoplar la cápsula Dragon 2 a la ISS usando el simulador de SpaceX.

In [ ]:
import time
import math
import threading
import queue
import matplotlib.pyplot as plt
from collections import deque
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from IPython.display import clear_output, display

In [ ]:
class DiscretePID:
    def __init__(self, Kp, Ki, Kd, Ts, u_min, u_max, tau=0.05):
        """
        Controlador PID en tiempo discreto.
        Incluye integración trapezoidal, derivación con filtro pasa-bajas y Anti-windup.
        """
        self.Kp = Kp
        self.Ki = Ki
        self.Kd = Kd
        self.Ts = Ts
        self.u_min = u_min
        self.u_max = u_max
        self.tau = tau

        self.e_prev = 0.0
        self.integral = 0.0
        self.derivative = 0.0

    def compute(self, setpoint, measurement):
        error = setpoint - measurement

        P = self.Kp * error
        integral_update = self.integral + self.Ki * self.Ts * 0.5 * (error + self.e_prev)
        
        D = ( (2 * self.tau - self.Ts) / (2 * self.tau + self.Ts) ) * self.derivative \
            + (2 * self.Kd / (2 * self.tau + self.Ts)) * (error - self.e_prev)

        self.derivative = D
        u = P + integral_update + D

        if u > self.u_max:
            u = self.u_max
            if error < 0:
                self.integral = integral_update
        elif u < self.u_min:
            u = self.u_min
            if error > 0:
                self.integral = integral_update
        else:
            self.integral = integral_update

        self.e_prev = error
        return u

class DeltaSigmaModulator:
    def __init__(self, threshold, output_mag):
        """
        Modulador Delta-Sigma de primer orden.
        """
        self.sigma = 0.0 
        self.threshold = threshold
        self.output_mag = output_mag

    def modulate(self, u_continuous):
        self.sigma += u_continuous

        if self.sigma >= self.threshold:
            self.sigma -= self.output_mag 
            return 1
        elif self.sigma <= -self.threshold:
            self.sigma += self.output_mag
            return -1
        else:
            return 0

In [ ]:
def setup_driver():
    options = webdriver.FirefoxOptions()
    options.add_argument('--headless') 
    driver = webdriver.Firefox(options=options)
    driver.get("https://iss-sim.spacex.com/")

    wait = WebDriverWait(driver, 15) # Wait time
    try:
        start_btn = wait.until(lambda d: d.find_element(By.ID, "begin-button"))
        start_btn.click()
        time.sleep(2) 
    except Exception as e:
        print("El botón Begin no se detectó o se omitió. Asumiendo que la simulación ya está corriendo.")

    return driver

def get_float_from_xpath(driver, xpath, remove_chars):
    try:
        text = driver.find_element(By.XPATH, xpath).text
        if len(text) > remove_chars:
            return float(text[:len(text)-remove_chars])
        return 0.0
    except:
        return 0.0

def read_sensors(driver):
    state = {}
    
    # Translación (X es Range, Y y Z son los crosshairs)
    state['x'] = get_float_from_xpath(driver, '//*[@id="range"]/div[2]', 2) 
    state['y'] = get_float_from_xpath(driver, '//*[@id="y-error"]', 2) 
    state['z'] = get_float_from_xpath(driver, '//*[@id="z-error"]', 2) 

    # Rotación
    state['roll'] = get_float_from_xpath(driver, '//*[@id="roll"]/div[1]', 1) 
    state['pitch'] = get_float_from_xpath(driver, '//*[@id="pitch"]/div[1]', 1) 
    state['yaw'] = get_float_from_xpath(driver, '//*[@id="yaw"]/div[1]', 1) 

    return state

In [ ]:
data_queue = queue.Queue()

def live_plot():
    """
    Se ejecuta en el HILO PRINCIPAL para evitar errores de Matplotlib con IPython.display.clear_output
    """
    plot_data = {
        't': deque(maxlen=50),
        'err_x': deque(maxlen=50),
        'err_y': deque(maxlen=50),
        'err_z': deque(maxlen=50),
        'u_x': deque(maxlen=50),
        'itae_acc': 0.0
    }

    while True:
        try:
            # wait for data
            new_data = None
            while not data_queue.empty():
                new_data = data_queue.get_nowait()
            
            if new_data == "STOP":
                print(f"Bucle de control terminado. ITAE Final: {plot_data['itae_acc']:.2f}")
                break
                
            if new_data:
                plot_data['t'].append(new_data['t'])
                plot_data['err_x'].append(new_data['err_x'])
                plot_data['err_y'].append(new_data['err_y'])
                plot_data['err_z'].append(new_data['err_z'])
                plot_data['u_x'].append(new_data['u_x'])
                plot_data['itae_acc'] = new_data['itae_acc']

                if len(plot_data['t']) > 0:
                    clear_output(wait=True)
                    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8,6))

                    t = list(plot_data['t'])
                    
                    ax1.plot(t, list(plot_data['err_x']), label='Error X (Range)')
                    ax1.plot(t, list(plot_data['err_y']), label='Error Y')
                    ax1.plot(t, list(plot_data['err_z']), label='Error Z')
                    ax1.set_title(f"Errores de Translación - ITAE: {plot_data['itae_acc']:.2f}")
                    ax1.set_ylabel("Metros (m)")
                    ax1.legend(loc="upper right")
                    ax1.grid(True)

                    ax2.step(t, list(plot_data['u_x']), label='u_x discreto (clics)', color='orange')
                    ax2.set_title("Acción de Control Discreta (Eje X)")
                    ax2.set_xlabel("Tiempo (s)")
                    ax2.set_ylabel("Pulsos")
                    ax2.legend(loc="upper right")
                    ax2.grid(True)
                    
                    plt.tight_layout()
                    plt.show()
                    plt.close(fig)

            time.sleep(0.5)
        except KeyboardInterrupt:
            print("Visualización interrumpida.")
            break

In [ ]:
def main_control_loop():
    try:
        driver = setup_driver()

        buttons = {
            'x_pos': driver.find_element(By.ID, "translate-forward-button"),
            'x_neg': driver.find_element(By.ID, "translate-backward-button"),
            'y_pos': driver.find_element(By.ID, "translate-left-button"), 
            'y_neg': driver.find_element(By.ID, "translate-right-button"),
            'z_pos': driver.find_element(By.ID, "translate-down-button"), 
            'z_neg': driver.find_element(By.ID, "translate-up-button"),

            'roll_pos': driver.find_element(By.ID, "roll-right-button"),
            'roll_neg': driver.find_element(By.ID, "roll-left-button"),
            'pitch_pos': driver.find_element(By.ID, "pitch-up-button"),
            'pitch_neg': driver.find_element(By.ID, "pitch-down-button"),
            'yaw_pos': driver.find_element(By.ID, "yaw-right-button"),
            'yaw_neg': driver.find_element(By.ID, "yaw-left-button")
        }
    except Exception as e:
        print(f"Error al inicializar los botones de la interfaz: {e}")
        data_queue.put("STOP")
        try:
            driver.quit()
        except:
            pass
        return

    Ts = 0.2

    pid_x = DiscretePID(Kp=0.08, Ki=0.001, Kd=0.5, Ts=Ts, u_min=-1.0, u_max=1.0)
    pid_y = DiscretePID(Kp=0.15, Ki=0.005, Kd=0.8, Ts=Ts, u_min=-1.0, u_max=1.0)
    pid_z = DiscretePID(Kp=0.15, Ki=0.005, Kd=0.8, Ts=Ts, u_min=-1.0, u_max=1.0)

    pid_roll = DiscretePID(Kp=0.5, Ki=0.01, Kd=2.0, Ts=Ts, u_min=-1.0, u_max=1.0)
    pid_pitch = DiscretePID(Kp=0.5, Ki=0.01, Kd=2.0, Ts=Ts, u_min=-1.0, u_max=1.0)
    pid_yaw = DiscretePID(Kp=0.5, Ki=0.01, Kd=2.0, Ts=Ts, u_min=-1.0, u_max=1.0)

    mod_x = DeltaSigmaModulator(threshold=0.5, output_mag=1.0)
    mod_y = DeltaSigmaModulator(threshold=0.5, output_mag=1.0)
    mod_z = DeltaSigmaModulator(threshold=0.5, output_mag=1.0)

    mod_roll = DeltaSigmaModulator(threshold=0.5, output_mag=1.0)
    mod_pitch = DeltaSigmaModulator(threshold=0.5, output_mag=1.0)
    mod_yaw = DeltaSigmaModulator(threshold=0.5, output_mag=1.0)

    sp = {'x': 0.0, 'y': 0.0, 'z': 0.0, 'roll': 0.0, 'pitch': 0.0, 'yaw': 0.0}

    print("Iniciando Acople Autónomo con Control PID (6 DOF)...")
    start_time = time.time()
    next_call = start_time
    
    itae_accumulator = 0.0

    try:
        while True:
            next_call += Ts
            sleep_time = next_call - time.time()
            if sleep_time > 0:
                time.sleep(sleep_time)

            state = read_sensors(driver)

            u_c_x = pid_x.compute(0.0, -state['x']) 
            u_c_y = pid_y.compute(sp['y'], state['y'])
            u_c_z = pid_z.compute(sp['z'], state['z'])

            u_c_roll = pid_roll.compute(sp['roll'], state['roll'])
            u_c_pitch = pid_pitch.compute(sp['pitch'], state['pitch'])
            u_c_yaw = pid_yaw.compute(sp['yaw'], state['yaw'])

            u_d_x = mod_x.modulate(u_c_x)
            u_d_y = mod_y.modulate(u_c_y)
            u_d_z = mod_z.modulate(u_c_z)

            u_d_roll = mod_roll.modulate(u_c_roll)
            u_d_pitch = mod_pitch.modulate(u_c_pitch)
            u_d_yaw = mod_yaw.modulate(u_c_yaw)

            try:
                if u_d_x == 1: buttons['x_pos'].click()
                elif u_d_x == -1: buttons['x_neg'].click()

                if u_d_y == 1: buttons['y_pos'].click()
                elif u_d_y == -1: buttons['y_neg'].click()

                if u_d_z == 1: buttons['z_pos'].click()
                elif u_d_z == -1: buttons['z_neg'].click()

                if u_d_roll == 1: buttons['roll_pos'].click()
                elif u_d_roll == -1: buttons['roll_neg'].click()

                if u_d_pitch == 1: buttons['pitch_pos'].click()
                elif u_d_pitch == -1: buttons['pitch_neg'].click()

                if u_d_yaw == 1: buttons['yaw_pos'].click()
                elif u_d_yaw == -1: buttons['yaw_neg'].click()
            except Exception as e:
                print(f"Error al hacer click: {e}")
                
            current_t = time.time() - start_time
            
            # ITAE calculation (Integral of Time-multiplied Absolute-Error)
            total_error = abs(-state['x']) + abs(state['y']) + abs(state['z']) + abs(state['roll']) + abs(state['pitch']) + abs(state['yaw'])
            itae_accumulator += current_t * total_error * Ts

            data_queue.put({
                't': current_t,
                'err_x': state['x'],
                'err_y': state['y'],
                'err_z': state['z'],
                'u_x': u_d_x,
                'itae_acc': itae_accumulator
            })

            if (state['x'] < 0.2 and abs(state['y']) < 0.2 and abs(state['z']) < 0.2 and
                abs(state['roll']) < 0.2 and abs(state['pitch']) < 0.2 and abs(state['yaw']) < 0.2):
                print("¡ACOPLE EXITOSO CON LA ISS!")
                data_queue.put("STOP")
                break

    except KeyboardInterrupt:
        print("Interrupción del usuario. Deteniendo PID.")
        data_queue.put("STOP")
    except Exception as e:
        print(f"Error en hilo de control: {e}")
        data_queue.put("STOP")
    finally:
        try:
            driver.quit() 
        except:
            pass

# Start control loop
control_thread = threading.Thread(target=main_control_loop)
control_thread.daemon = True
control_thread.start()

# Start live plotting
live_plot()